# Zuse on Amazon SageMaker

Zuse is Bineric's European decision model. It reads a document and answers fixed-choice questions about it in one pass
(yes/no, choice among options, position on a scale), each with a calibrated probability. It does not generate text.

This notebook deploys Zuse from your AWS Marketplace subscription, sends real-time requests, runs a batch transform and cleans up.
Prerequisites: a subscription to **Zuse 27B** or **Zuse 9B** in AWS Marketplace, and a SageMaker execution role.

In [ ]:
import json, boto3, sagemaker
from sagemaker import ModelPackage, get_execution_role

# The model package ARN from your subscription (AWS Marketplace > Manage subscriptions > Zuse > Launch > SageMaker console).
MODEL_PACKAGE_ARN = "<your Zuse model package ARN>"
INSTANCE_TYPE = "ml.g5.2xlarge"   # or ml.g6.2xlarge / ml.g6e.xlarge; the 9B also runs on ml.g5.xlarge
ENDPOINT_NAME = "zuse"

session = sagemaker.Session()
role = get_execution_role()

## 1. Deploy a real-time endpoint

Takes 10-20 minutes. If AWS has no free capacity for the instance type (`InsufficientInstanceCapacity`), pick another supported type.

In [ ]:
model = ModelPackage(role=role, model_package_arn=MODEL_PACKAGE_ARN, sagemaker_session=session)
model.deploy(initial_instance_count=1, instance_type=INSTANCE_TYPE, endpoint_name=ENDPOINT_NAME)

## 2. Ask questions about a document

A request is a `state` (the document) and a map of named questions. Question types:

| type | asks | criteria | answer |
|---|---|---|---|
| `noul` | yes/no | none | `noul`: probability of yes |
| `choice` | one option of up to 255 | `{"name": "description"}` | `choice`, `confidence`, `probabilities` |
| `score` | a position on an ordered scale (2-10 levels) | `["level 0", "level 1", ...]` | `score` (expected level), `legend`, `probabilities` |

In [ ]:
runtime = boto3.client("sagemaker-runtime")

def ask(body):
    r = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType="application/json", Body=json.dumps(body))
    return json.loads(r["Body"].read())

body = {
  "state": "Hei, jeg vil avslutte kredittkortet mitt og flytte boliglånet til en annen bank. Kan dere sende meg innfrielsesbeløpet?",
  "questions": {
    "intent": {"type": "choice", "instructions": "What does the customer want?",
               "criteria": {"a": "Close the customer relationship or move the loan", "b": "A question about a card transaction", "c": "Something else"}},
    "asks_for_document": {"type": "noul", "instructions": "Does the customer ask the bank to send them something?"}}}
print(json.dumps(ask(body), indent=2, ensure_ascii=False))

## 3. Use the probabilities

Automate the answers whose probability clears a threshold and send the rest to a person. Choose the threshold on a few hundred of your
own labelled examples.

In [ ]:
THRESHOLD = 0.9
answer = ask(body)["answers"]["intent"]
if answer["confidence"] >= THRESHOLD:
    print("automate:", answer["choice"])
else:
    print("send to a person; best guess", answer["choice"], answer["probabilities"])

## 4. Batch transform

One JSON request per line. `samples/input.jsonl` in this repository has three: an English contract, a Norwegian bank message and a
German insurance claim.

In [ ]:
input_s3 = session.upload_data("samples/input.jsonl", key_prefix="zuse/batch-input")
transformer = model.transformer(instance_count=1, instance_type=INSTANCE_TYPE, strategy="SingleRecord", assemble_with="Line",
                                accept="application/json", output_path=f"s3://{session.default_bucket()}/zuse/batch-output")
transformer.transform(input_s3, content_type="application/json", split_type="Line")
transformer.wait()
print(session.read_s3_file(session.default_bucket(), "zuse/batch-output/input.jsonl.out"))

## 5. Clean up

The endpoint bills by the hour while it runs.

In [ ]:
model.sagemaker_session.delete_endpoint(ENDPOINT_NAME)
model.sagemaker_session.delete_endpoint_config(ENDPOINT_NAME)
model.delete_model()